In [17]:
from mutagen.flac import FLAC
import pandas as pd
import os

ruta_csv = "data/dataset_musica.csv"
carpeta_audio = "audio/"
datos_nuevos = []

rutas_existentes = set()
proximo_id = 1

if os.path.exists(ruta_csv):
    df_existente = pd.read_csv(ruta_csv)
    if "Audio_Path" in df_existente.columns:
        rutas_existentes = set(df_existente["Audio_Path"].dropna())
    if "ID" in df_existente.columns and not df_existente.empty:
        proximo_id = int(df_existente["ID"].max()) + 1
else:
    df_existente = pd.DataFrame()

if os.path.exists(carpeta_audio):
    for archivo in os.listdir(carpeta_audio):
        if archivo.endswith(".flac"):
            ruta_completa = os.path.join(carpeta_audio, archivo)

            if ruta_completa in rutas_existentes:
                continue

            audio = FLAC(ruta_completa)

            letra_extraida = audio.get("lyrics")
            letra_texto = letra_extraida[0] if letra_extraida else "Sin letra disponible"

            datos_nuevos.append({
                "ID": proximo_id,
                "Titulo": audio.get("title", [archivo.replace(".flac", "")])[0],
                "Artista": audio.get("artist", ["Desconocido"])[0],
                "Album": audio.get("album", ["Desconocido"])[0],
                "Genero": audio.get("genre", ["Desconocido"])[0],
                "Año": audio.get("date", ["Desconocido"])[0],
                "Letra": letra_texto,
                "Audio_Path": ruta_completa
            })
            proximo_id += 1

if datos_nuevos:
    df_nuevos = pd.DataFrame(datos_nuevos)
    df_final = pd.concat([df_existente, df_nuevos], ignore_index=True)
    df_final.to_csv(ruta_csv, index=False, encoding="utf-8-sig")
    print(f"¡Éxito! Se agregaron {len(datos_nuevos)} canciones nuevas al dataset.")
else:
    print("No se encontraron canciones nuevas. Tu dataset está al día.")

No se encontraron canciones nuevas. Tu dataset está al día.


In [18]:
df_musica = pd.read_csv("data/dataset_musica.csv")
df_musica.head()

,ID,Titulo,Artista,Album,Genero,Año,Letra,Audio_Path
0,1,NUEVAYoL,Bad Bunny,DeBÍ TiRAR MáS FOToS,World,2025-01-04,Nueva Yol\r\nSi te quieres divertir\r\nCon enc...,audio/01. Bad Bunny - NUEVAYoL.flac
1,2,LUZ DE LUNA,HUMBE,DUEÑO DEL CIELO,Desconocido,2025-12-06,"Yo no sé, no me siento a salvo\r\n¿Será que me...",audio/01. HUMBE - LUZ DE LUNA.flac
2,3,BLOOD.,Kendrick Lamar,DAMN.,Hip-Hop/Rap,2017-04-13,Is it wickedness?\r\nIs it weakness?\r\nYou de...,audio/01. Kendrick Lamar - BLOOD..flac
3,4,St. Chroma,"Tyler, The Creator",CHROMAKOPIA,Hip-Hop/Rap,2024-10-27,"you are the light\r\nit's not on you, it's in ...","audio/01. Tyler, The Creator - St. Chroma.flac"
4,5,VOY A LLeVARTE PA PR,Bad Bunny,DeBÍ TiRAR MáS FOToS,World,2025-01-04,"Acho, PR es otra cosa\r\nYo la conocí en Miami...",audio/02. Bad Bunny - VOY A LLeVARTE PA PR.flac


In [19]:
%%writefile app.py
import streamlit as st
import pandas as pd
import os
#Diseño 
st.set_page_config(
    page_title="Music AI Bot",
    page_icon="🎵",
    layout="centered"
)

st.markdown("""
    <style>
    .main .block-container { max-width: 750px; padding-top: 2rem; }
    div.stChatInput { position: fixed; bottom: 3rem; max-width: 750px; z-index: 99; }
    </style>
""", unsafe_allow_html=True)

with st.sidebar:
    st.title("Módulos de IA")
    st.write("Panel para interactuar con las funciones de audio.")
    st.markdown("---")
    st.subheader("Reconocimiento de Audio")
    
    archivo_audio = st.file_uploader(
        "Sube un fragmento de audio (.flac / .wav)", 
        type=["flac", "wav"]
    )
    
    if archivo_audio is not None:
        st.info("Audio recibido. Procesando frecuencias...")

st.title("Music AI")
st.caption("Chatbot inteligente de recomendación, letras y reconocimiento musical.")

if "messages" not in st.session_state:
    st.session_state.messages = [
        {"role": "assistant", "content": "¡Hola! Soy tu asistente musical. Puedes pedirme recomendaciones por género o artista, o escribir una frase de una canción para buscar su letra. ¿Qué escuchamos hoy?"}
    ]

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.write(message["content"])

if prompt := st.chat_input("Pregúntale algo a tu asistente musical..."):

    with st.chat_message("user"):
        st.write(prompt)
    st.session_state.messages.append({"role": "user", "content": prompt})

    with st.chat_message("assistant"):
        mensaje_espera = st.empty()
        mensaje_espera.markdown("*Pensando...*")
        
        # Simulación de respuesta conectada al dataset
        respuesta_bot = f"Recibí tu petición: '{prompt}'. Aquí se llamará al modelo de Machine Learning correspondiente usando el archivo dataset_musica.csv."
        mensaje_espera.write(respuesta_bot)
        
    st.session_state.messages.append({"role": "assistant", "content": respuesta_bot})
    #Usar "streamlit run app.py" en la terminal para iniciar la aplicación.

Overwriting app.py
